In [13]:
import pyodbc
import pandas as pd
from sqlalchemy import create_engine

# connecting to sql server using windows authentication
conn = pyodbc.connect(
    "DRIVER={SQL Server};"
    "SERVER=ISHANKPC\\SQLEXPRESS;"
    "DATABASE=mutualfund_analysis;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()
print("connected to sql server")

# sqlalchemy engine — pandas needs this for read_sql
engine = create_engine(
    "mssql+pyodbc://ISHANKPC\\SQLEXPRESS/mutualfund_analysis"
    "?driver=SQL+Server&trusted_connection=yes"
)
print("sqlalchemy engine ready")
print()

# drop and recreate all 3 tables fresh
cursor.execute("""
    IF OBJECT_ID('fund_metrics', 'U') IS NOT NULL
        DROP TABLE fund_metrics
    CREATE TABLE fund_metrics (
        fund_name       VARCHAR(100),
        category        VARCHAR(50),
        cagr_5y         DECIMAL(6,2),
        cagr_1y         DECIMAL(6,2),
        volatility      DECIMAL(6,2),
        sharpe_ratio    DECIMAL(6,2),
        max_drawdown    DECIMAL(6,2)
    )
""")

cursor.execute("""
    IF OBJECT_ID('sip_results', 'U') IS NOT NULL
        DROP TABLE sip_results
    CREATE TABLE sip_results (
        fund_name       VARCHAR(100),
        category        VARCHAR(50),
        total_invested  DECIMAL(12,2),
        current_value   DECIMAL(12,2),
        profit          DECIMAL(12,2),
        sip_returns     DECIMAL(6,2),
        total_units     DECIMAL(12,3),
        latest_nav      DECIMAL(10,4)
    )
""")

cursor.execute("""
    IF OBJECT_ID('fund_rankings', 'U') IS NOT NULL
        DROP TABLE fund_rankings
    CREATE TABLE fund_rankings (
        rank_no         INT,
        fund_name       VARCHAR(100),
        category        VARCHAR(50),
        cagr_5y         DECIMAL(6,2),
        sharpe_ratio    DECIMAL(6,2),
        sip_returns     DECIMAL(6,2),
        max_drawdown    DECIMAL(6,2)
    )
""")

conn.commit()
print("3 tables created fresh:")
print("  - fund_metrics")
print("  - sip_results")
print("  - fund_rankings")

connected to sql server
sqlalchemy engine ready

3 tables created fresh:
  - fund_metrics
  - sip_results
  - fund_rankings


In [14]:
# read the csvs we saved from earlier notebooks
comparison_df = pd.read_csv('../data/fund_comparison.csv')
sip_df        = pd.read_csv('../data/sip_comparison.csv')
final_df      = pd.read_csv('../data/final_rankings.csv')

# load fund metrics
for _, row in comparison_df.iterrows():
    cursor.execute(
        "INSERT INTO fund_metrics VALUES (?, ?, ?, ?, ?, ?, ?)",
        row['Fund Name'], row['Category'], row['CAGR 5Y (%)'],
        row['CAGR 1Y (%)'], row['Volatility (%)'],
        row['Sharpe Ratio'], row['Max Drawdown (%)']
    )
print(f"fund_metrics   — {len(comparison_df)} rows loaded")

# load sip results
for _, row in sip_df.iterrows():
    cursor.execute(
        "INSERT INTO sip_results VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
        row['Fund Name'], row['Category'], row['Total Invested'],
        row['Current Value'], row['Profit'],
        row['Returns (%)'], row['Total Units'], row['Latest NAV']
    )
print(f"sip_results    — {len(sip_df)} rows loaded")

# load final rankings
for _, row in final_df.iterrows():
    cursor.execute(
        "INSERT INTO fund_rankings VALUES (?, ?, ?, ?, ?, ?, ?)",
        row['Rank'], row['Fund Name'], row['Category'],
        row['CAGR 5Y (%)'], row['Sharpe Ratio'],
        row['SIP Returns (%)'], row['Max Drawdown (%)']
    )
print(f"fund_rankings  — {len(final_df)} rows loaded")

conn.commit()
print("\nall data in sql server — ready to query")

fund_metrics   — 33 rows loaded
sip_results    — 33 rows loaded
fund_rankings  — 33 rows loaded

all data in sql server — ready to query


In [15]:
# helper function so we dont repeat print statements for every query
def run_query(title, query):
    print("=" * 65)
    print(title)
    print("=" * 65)
    df = pd.read_sql(query, engine)
    print(df.to_string(index=False))
    print()
    return df

In [20]:
# which category performed best overall?
# grouped by category, averaging all key metrics

q1 = run_query(
    "Q1. CATEGORY PERFORMANCE SUMMARY",
    """
    SELECT
        category,
        COUNT(*)                     AS total_funds,
        ROUND(AVG(cagr_5y), 2)      AS avg_cagr,
        ROUND(AVG(sharpe_ratio), 2)  AS avg_sharpe,
        ROUND(AVG(volatility), 2)    AS avg_volatility,
        ROUND(AVG(max_drawdown), 2)  AS avg_drawdown
    FROM fund_metrics
    GROUP BY category
    ORDER BY avg_cagr DESC
    """
)

Q1. CATEGORY PERFORMANCE SUMMARY
  category  total_funds  avg_cagr  avg_sharpe  avg_volatility  avg_drawdown
 Small Cap            9     24.78        0.98           17.90        -37.47
   Mid Cap           10     20.56        0.80           16.49        -31.60
 Large Cap            6     16.55        0.64           16.88        -36.60
Index Fund            8     14.88        0.52           17.92        -35.93



In [16]:
# query 2 — which active funds beat the best index fund?
q2 = run_query(
    "Q2. ACTIVE FUNDS BEATING THE BEST INDEX FUND (22.43% CAGR)",
    """
    SELECT
        fund_name,
        category,
        cagr_5y,
        sharpe_ratio
    FROM fund_metrics
    WHERE category != 'Index Fund'
      AND cagr_5y > (
            SELECT MAX(cagr_5y)
            FROM fund_metrics
            WHERE category = 'Index Fund'
          )
    ORDER BY cagr_5y DESC
    """
)

Q2. ACTIVE FUNDS BEATING THE BEST INDEX FUND (22.43% CAGR)
                       fund_name  category  cagr_5y  sharpe_ratio
            Quant Small Cap Fund Small Cap    35.54          1.29
     Nippon India Small Cap Fund Small Cap    27.26          1.06
    Canara Robeco Small Cap Fund Small Cap    26.21          1.04
              Quant Mid Cap Fund   Mid Cap    25.70          0.96
          Edelweiss Mid Cap Fund   Mid Cap    25.47          0.98
Nippon India Growth Mid Cap Fund   Mid Cap    24.20          0.95
              DSP Small Cap Fund Small Cap    24.13          0.96
       Invesco India Midcap Fund   Mid Cap    23.94          0.94
             Tata Small Cap Fund Small Cap    22.74          0.90
       Motilal Oswal Midcap Fund   Mid Cap    22.47          0.83



In [17]:
# query 3 — top 2 funds per category by sip returns
q3 = run_query(
    "Q3. TOP 2 FUNDS PER CATEGORY BY SIP RETURNS",
    """
    SELECT fund_name, category, sip_returns, current_value, profit
    FROM (
        SELECT *,
               RANK() OVER (PARTITION BY category ORDER BY sip_returns DESC) AS rnk
        FROM sip_results
    ) ranked
    WHERE rnk <= 2
    ORDER BY category, sip_returns DESC
    """
)

Q3. TOP 2 FUNDS PER CATEGORY BY SIP RETURNS
                                fund_name   category  sip_returns  current_value    profit
Motilal Oswal Nifty Midcap 150 Index Fund Index Fund       118.85      667488.83 362488.83
     DSP Nifty 50 Equal Weight Index Fund Index Fund        84.53      562811.49 257811.49
                      HDFC Flexi Cap Fund  Large Cap       109.06      637618.85 332618.85
              Parag Parikh Flexi Cap Fund  Large Cap        92.53      587220.12 282220.12
                   Edelweiss Mid Cap Fund    Mid Cap       144.23      744895.30 439895.30
         Nippon India Growth Mid Cap Fund    Mid Cap       139.53      730567.44 425567.44
                     Quant Small Cap Fund  Small Cap       174.97      838659.56 533659.56
              Nippon India Small Cap Fund  Small Cap       145.87      749917.20 444917.20



In [18]:
# query 4 — sweet spot funds: high return, low drawdown
# looking for funds where cagr is above average but drawdown is below average
q4 = run_query(
    "Q4. SWEET SPOT FUNDS — HIGH RETURN, LOW DRAWDOWN",
    """
    SELECT
        fund_name,
        category,
        cagr_5y,
        max_drawdown,
        sharpe_ratio
    FROM fund_metrics
    WHERE cagr_5y     > (SELECT AVG(cagr_5y) FROM fund_metrics)
      AND max_drawdown > (SELECT AVG(max_drawdown) FROM fund_metrics)
    ORDER BY sharpe_ratio DESC++
    """
)

Q4. SWEET SPOT FUNDS — HIGH RETURN, LOW DRAWDOWN
                               fund_name   category  cagr_5y  max_drawdown  sharpe_ratio
             Parag Parikh Flexi Cap Fund  Large Cap    20.39        -31.20          0.98
                      Quant Mid Cap Fund    Mid Cap    25.70        -33.43          0.96
               Invesco India Midcap Fund    Mid Cap    23.94        -34.09          0.94
                     Axis Small Cap Fund  Small Cap    22.34        -34.64          0.93
                      SBI Small Cap Fund  Small Cap    20.54        -33.85          0.88
Nippon India Nifty Midcap 150 Index Fund Index Fund    19.90        -21.44          0.80



In [19]:
# query 5 — joining metrics and sip results to get full picture
q5 = run_query(
    "Q5. FULL FUND SCORECARD — METRICS + SIP (TOP 10)",
    """
    SELECT TOP 10
        m.fund_name,
        m.category,
        m.cagr_5y,
        m.sharpe_ratio,
        s.sip_returns,
        s.current_value,
        s.profit
    FROM fund_metrics m
    JOIN sip_results s ON m.fund_name = s.fund_name
    ORDER BY m.sharpe_ratio DESC
    """
)

Q5. FULL FUND SCORECARD — METRICS + SIP (TOP 10)
                       fund_name  category  cagr_5y  sharpe_ratio  sip_returns  current_value    profit
            Quant Small Cap Fund Small Cap    35.54          1.29       174.97      838659.56 533659.56
     Nippon India Small Cap Fund Small Cap    27.26          1.06       145.87      749917.20 444917.20
    Canara Robeco Small Cap Fund Small Cap    26.21          1.04       114.68      654770.23 349770.23
     Parag Parikh Flexi Cap Fund Large Cap    20.39          0.98        92.53      587220.12 282220.12
          Edelweiss Mid Cap Fund   Mid Cap    25.47          0.98       144.23      744895.30 439895.30
              DSP Small Cap Fund Small Cap    24.13          0.96       120.59      672812.92 367812.92
              Quant Mid Cap Fund   Mid Cap    25.70          0.96       115.53      657375.65 352375.65
Nippon India Growth Mid Cap Fund   Mid Cap    24.20          0.95       139.53      730567.44 425567.44
       Invesco 

In [21]:
# exporting clean summary tables for power bi
# power bi will connect directly to these csvs

# full fund scorecard — all metrics in one place
scorecard = pd.read_sql("""
    SELECT
        m.fund_name,
        m.category,
        m.cagr_5y,
        m.cagr_1y,
        m.volatility,
        m.sharpe_ratio,
        m.max_drawdown,
        s.sip_returns,
        s.current_value,
        s.profit,
        r.rank_no
    FROM fund_metrics m
    JOIN sip_results s  ON m.fund_name = s.fund_name
    JOIN fund_rankings r ON m.fund_name = r.fund_name
""", engine)

scorecard.to_csv('../data/powerbi_scorecard.csv', index=False)
print(f"powerbi_scorecard.csv saved — {len(scorecard)} rows")

# category summary for kpi cards and bar charts
cat_summary = pd.read_sql("""
    SELECT
        category,
        COUNT(*)                     AS total_funds,
        ROUND(AVG(cagr_5y), 2)      AS avg_cagr,
        ROUND(MAX(cagr_5y), 2)      AS best_cagr,
        ROUND(AVG(sharpe_ratio), 2)  AS avg_sharpe,
        ROUND(AVG(volatility), 2)    AS avg_volatility,
        ROUND(AVG(max_drawdown), 2)  AS avg_drawdown
    FROM fund_metrics
    GROUP BY category
""", engine)

cat_summary.to_csv('../data/powerbi_category.csv', index=False)
print(f"powerbi_category.csv saved — {len(cat_summary)} rows")

print("\nboth files saved to data/ folder — ready for power bi")

powerbi_scorecard.csv saved — 33 rows
powerbi_category.csv saved — 4 rows

both files saved to data/ folder — ready for power bi
